# 🩺 Predictor de Hipotensión Intradialítica
**DIALYSIS-DATA Consulting**

---

**Objetivo:** Predecir el riesgo de hipotensión intradialítica mediante regresión logística, usando variables clínicas y de máquina capturables en cualquier unidad de hemodiálisis.

**Dataset:** MIMIC (Medical Information Mart for Intensive Care) — uso libre para investigación.  
**N:** 12,404 sesiones | **Eventos:** 2,481 hipotensiones (20%)  
**AUC:** 0.714

**Estándares de referencia:** KDIGO, AAMI, NOM-003 Hemodiálisis

---
> ⚠️ Este modelo fue entrenado en población MIMIC (EE.UU.). Para uso clínico en México se recomienda recalibración con datos locales.

## 1. Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve,
    auc,
    recall_score,
    precision_score,
    f1_score,
    precision_recall_curve
)
import pickle

## 2. Carga de datos

In [ ]:
# Si usas Google Colab:
# from google.colab import files
# uploaded = files.upload()

df = pd.read_excel('11.xlsx')

print(f'Shape: {df.shape}')
print(f'Variables: {df.columns.tolist()}')
df.head(3)

## 3. Limpieza y validación de datos

In [ ]:
# Verificar valores nulos
print('=== VALORES NULOS ===')
print(df.isnull().sum())

# Renombrar columnas clave
df = df.rename(columns={
    'D_PesoPre'    : 'peso_pre',
    'D_PesoPost'   : 'peso_post',
    'D_Edad'       : 'edad',
    'D_Hipotension': 'hipotension'
})

# Eliminar registros con peso fuera de rango clínico
df = df[(df['peso_pre'] > 30) & (df['peso_pre'] < 200)]

# Convertir variable objetivo a numérico
df['hipotension'] = df['hipotension'].map({'NO': 0, 'SI': 1})

print(f'\n=== BALANCE DE CLASES ===')
print(df['hipotension'].value_counts())
print(df['hipotension'].value_counts(normalize=True).round(3))

## 4. Ingeniería de variables clínicas

Se crean variables derivadas con valor clínico interpretable:

| Variable | Fórmula | Referencia clínica |
|---|---|---|
| `uf_total` | peso_pre - peso_post | Volumen total extraído (L) |
| `uf_ml_kg_h` | (uf_total × 1000) / (peso_pre × 3h) | Tasa UF — KDOQI: riesgo si > 13 ml/kg/h |
| `delta_presion` | TAS paciente - Presión arterial máquina | Diferencial hemodinámico paciente vs circuito |
| `stress_hemodinamico` | uf_ml_kg_h / TAS | Índice compuesto de carga hemodinámica |

In [ ]:
# Nota: tiempo_sesion fijo en 3h (unidades ambulatorias CDMX)
# Para otras duraciones, modificar el divisor en uf_ml_kg_h
TIEMPO_SESION_HORAS = 3

df['uf_total']           = df['peso_pre'] - df['peso_post']
df['uf_ml_kg_h']         = (df['uf_total'] * 1000) / (df['peso_pre'] * TIEMPO_SESION_HORAS)
df['delta_presion']      = df['D_TAS'] - df['D_PresionArterial']
df['stress_hemodinamico']= df['uf_ml_kg_h'] / df['D_TAS']

print('Variables clínicas creadas correctamente.')
df[['uf_total','uf_ml_kg_h','delta_presion','stress_hemodinamico']].describe().round(2)

## 5. Selección de variables predictoras

Variables agrupadas por dominio clínico:

In [ ]:
FEATURES = [
    # Demográficas
    'edad',
    'peso_pre',

    # Hemodinámicas del paciente
    'D_TAS',
    'D_TAD',

    # Parámetros de máquina
    'D_PresionArterial',
    'D_PresionVenosa',
    'D_PTM',
    'D_FlujoSangre',

    # Balance de volumen
    'D_UF',

    # Variables clínicas derivadas
    'uf_ml_kg_h',
    'delta_presion',
    'stress_hemodinamico'
]

X = df[FEATURES]
y = df['hipotension']

print(f'Features: {len(FEATURES)}')
print(f'N sesiones: {len(X)}')

## 6. División de datos y escalado

> ⚠️ **Orden crítico:** primero dividir, luego escalar.  
> El scaler se ajusta SOLO con datos de entrenamiento para evitar data leakage.

In [ ]:
# 1. Dividir primero
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2. Escalar después (fit solo en train)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)  # solo transform, no fit

print(f'Train: {X_train_s.shape} | Test: {X_test_s.shape}')

## 7. Entrenamiento del modelo

In [ ]:
modelo = LogisticRegression(max_iter=1000, random_state=42)
modelo.fit(X_train_s, y_train)

y_prob = modelo.predict_proba(X_test_s)[:, 1]

roc_auc = roc_auc_score(y_test, y_prob)
print(f'AUC: {roc_auc:.3f}')

## 8. Coeficientes e interpretación clínica

Con escalado, los Odds Ratio son comparables entre variables.

In [ ]:
coef_df = pd.DataFrame({
    'Variable'   : FEATURES,
    'Coeficiente': modelo.coef_[0],
    'Odds_Ratio' : np.exp(modelo.coef_[0])
}).sort_values('Odds_Ratio', ascending=False)

print('=== ODDS RATIO (escalados) ===')
print(coef_df.to_string(index=False))
print(f'\nIntercepto: {modelo.intercept_[0]:.6f}')

**Interpretación clínica de los predictores principales:**

| Variable | OR | Lectura clínica |
|---|---|---|
| D_TAS | 1.79 | TAS basal alta → mayor caída relativa (hipertensos crónicos) |
| stress_hemodinamico | 1.49 | Índice compuesto: UF agresiva + TAS baja = riesgo alto |
| delta_presion | 1.35 | Diferencial paciente vs circuito: marcador de inestabilidad |
| D_UF | 1.34 | Mayor volumen de extracción, mayor riesgo |
| D_FlujoSangre | 0.72 | Flujo alto = sesión más eficiente, menor estrés |
| uf_ml_kg_h | 0.70 | Efecto compensado por stress_hemodinamico (colinealidad parcial) |

## 9. Selección de threshold clínico

En hemodiálisis, **falsos negativos son más costosos que falsos positivos**.  
Priorizamos recall (sensibilidad) sobre precisión.

In [ ]:
resultados = []
for t in [0.30, 0.25, 0.23, 0.20, 0.15, 0.14]:
    yp = (y_prob > t).astype(int)
    resultados.append({
        'threshold': t,
        'recall'   : round(recall_score(y_test, yp), 3),
        'precision': round(precision_score(y_test, yp), 3),
        'f1'       : round(f1_score(y_test, yp), 3)
    })

pd.DataFrame(resultados)

## 10. Semáforo clínico de riesgo

In [ ]:
def clasificar_riesgo(p):
    """
    Clasifica probabilidad de hipotensión en nivel de riesgo.
    Thresholds basados en optimización de recall clínico.
    
    ALTO  (p > 0.20): Intervención preventiva recomendada
    MEDIO (p > 0.14): Monitoreo frecuente
    BAJO  (p <= 0.14): Seguimiento estándar
    """
    if p > 0.20:
        return 'ALTO'
    elif p > 0.14:
        return 'MEDIO'
    else:
        return 'BAJO'

# Aplicar a dataset completo
y_prob_full = modelo.predict_proba(scaler.transform(X))[:, 1]
df['probabilidad'] = y_prob_full
df['riesgo']       = df['probabilidad'].apply(clasificar_riesgo)

print('=== DISTRIBUCIÓN DE RIESGO ===')
print(df['riesgo'].value_counts(normalize=True).round(3))

print('\n=== VALIDACIÓN: RIESGO vs EVENTO REAL ===')
print(pd.crosstab(df['riesgo'], df['hipotension'], normalize='index').round(3))

## 11. Curva ROC

In [ ]:
fpr, tpr, thresholds_roc = roc_curve(y_test, y_prob)
roc_auc_val = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='#0ea5e9', lw=2, label=f'AUC = {roc_auc_val:.3f}')
plt.plot([0,1], [0,1], linestyle='--', color='gray')

# Marcar thresholds clínicos
for t, color, label in [(0.20, 'red', 'Alto (0.20)'), (0.14, 'orange', 'Medio (0.14)')]:
    idx = np.argmin(np.abs(thresholds_roc - t))
    plt.scatter(fpr[idx], tpr[idx], color=color, zorder=5, s=80, label=label)

plt.xlabel('Tasa de Falsos Positivos')
plt.ylabel('Sensibilidad (Recall)')
plt.title('Curva ROC — Predictor de Hipotensión Intradialítica\nDIALYSIS-DATA Consulting')
plt.legend()
plt.tight_layout()
plt.savefig('curva_roc.png', dpi=150)
plt.show()

## 12. Exportar modelo

In [ ]:
# Guardar modelo Y scaler juntos (ambos son necesarios para predecir)
artefactos = {
    'modelo' : modelo,
    'scaler' : scaler,
    'features': FEATURES
}

with open('modelo_hipotension.pkl', 'wb') as f:
    pickle.dump(artefactos, f)

print('Modelo exportado: modelo_hipotension.pkl')
print(f'Contiene: modelo, scaler y lista de {len(FEATURES)} features')

# Si usas Google Colab, descomenta para descargar:
# from google.colab import files
# files.download('modelo_hipotension.pkl')

## 13. Función de predicción para uso clínico

In [ ]:
def predecir_riesgo(datos_paciente: dict) -> dict:
    """
    Predice riesgo de hipotensión para una sesión.
    
    Parámetros (dict):
        edad, peso_pre, D_TAS, D_TAD, D_PresionArterial,
        D_PresionVenosa, D_PTM, D_FlujoSangre, D_UF,
        peso_post (para calcular features derivadas)
    
    Retorna:
        dict con probabilidad, riesgo y variables calculadas
    """
    d = datos_paciente.copy()

    # Calcular features clínicas derivadas
    uf_total = d['peso_pre'] - d['peso_post']
    d['uf_ml_kg_h']          = (uf_total * 1000) / (d['peso_pre'] * 3)
    d['delta_presion']       = d['D_TAS'] - d['D_PresionArterial']
    d['stress_hemodinamico'] = d['uf_ml_kg_h'] / d['D_TAS']
    d['D_UF']                = d.get('D_UF', uf_total)

    # Armar vector de features
    X_nuevo = pd.DataFrame([{f: d[f] for f in FEATURES}])
    X_nuevo_s = scaler.transform(X_nuevo)

    prob = modelo.predict_proba(X_nuevo_s)[0, 1]
    nivel = clasificar_riesgo(prob)

    return {
        'probabilidad'        : round(prob, 3),
        'riesgo'              : nivel,
        'uf_ml_kg_h'          : round(d['uf_ml_kg_h'], 2),
        'stress_hemodinamico' : round(d['stress_hemodinamico'], 4),
        'alerta_uf_kdoqi'     : d['uf_ml_kg_h'] > 13
    }


# --- EJEMPLO DE USO ---
paciente_ejemplo = {
    'edad'             : 68,
    'peso_pre'         : 75.0,
    'peso_post'        : 72.5,
    'D_TAS'            : 145,
    'D_TAD'            : 85,
    'D_PresionArterial': 120,
    'D_PresionVenosa'  : 180,
    'D_PTM'            : 25,
    'D_FlujoSangre'    : 350,
    'D_UF'             : 2.5
}

resultado = predecir_riesgo(paciente_ejemplo)
print('=== RESULTADO DE PREDICCIÓN ===')
for k, v in resultado.items():
    print(f'{k}: {v}')

---
## Resumen del modelo

| Métrica | Valor |
|---|---|
| Algoritmo | Regresión Logística |
| AUC | 0.714 |
| Recall (t=0.14) | 0.810 |
| Recall (t=0.20) | 0.635 |
| N entrenamiento | 9,923 sesiones |
| N test | 2,481 sesiones |
| Dataset | MIMIC (uso libre) |

**Limitaciones:**
- Tiempo de sesión fijo en 3h (válido para unidades ambulatorias CDMX)
- Entrenado en población MIMIC — requiere validación en población mexicana
- Colinealidad parcial entre `D_UF` y `uf_ml_kg_h`

**Próximos pasos:**
- Recolección de datos por unidad para recalibración local
- Construcción de dataset nacional de hemodiálisis
- Validación prospectiva

---
*DIALYSIS-DATA Consulting — dialysis.data.consulting@gmail.com*